# v1 Stephen Curry career box scores

Research notebook to pull all regular-season Stephen Curry game-level box score rows across his full NBA career.

Context:
- Stephen Curry has played 17 seasons with Golden State.
- Reference checkpoints: 1,065 regular-season games and career averages near 24.8 PTS / 6.3 AST / 4.7 REB.

Goal:
- Resolve Curry's NBA player ID from `nba_api`
- Build full season list from his first to last season
- Pull every regular-season game log season-by-season
- Combine into one DataFrame for downstream modeling and validation

In [1]:
from __future__ import annotations

import ssl
import time
from pathlib import Path

import pandas as pd
import requests
import urllib3

# Repo-standard SSL workaround for nba_api/stats.nba.com calls
# NOTE: this must be set before importing nba_api endpoints.
ssl._create_default_https_context = ssl._create_unverified_context
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

_original_request = requests.Session.request


def _patched_request(self, *args, **kwargs):
    kwargs["verify"] = False
    return _original_request(self, *args, **kwargs)


requests.Session.request = _patched_request

from nba_api.stats.static import players
from nba_api.stats.endpoints import commonplayerinfo
from nba_api.stats.endpoints import playergamelog

In [2]:
PLAYER_NAME = "Stephen Curry"
SEASON_TYPE = "Regular Season"
RATE_LIMIT_SLEEP_SECONDS = 0.6

In [3]:
def get_player_id_exact(full_name: str) -> int:
    """Return exact NBA API player id for a single player name match."""
    matches = [p for p in players.get_players() if p["full_name"] == full_name]
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one match for {full_name}, found {len(matches)}")
    return int(matches[0]["id"])


def build_season_strings(from_year: int, to_year: int) -> list[str]:
    """Build nba_api season strings like 2009-10."""
    return [f"{y}-{str(y + 1)[-2:]}" for y in range(from_year, to_year + 1)]


def fetch_career_regular_season_logs(player_id: int, season_strings: list[str]) -> pd.DataFrame:
    """Fetch one regular-season game log DataFrame per season and combine."""
    season_frames = []
    for i, season in enumerate(season_strings, start=1):
        print(f"[{i}/{len(season_strings)}] fetching {season}")
        frame = playergamelog.PlayerGameLog(
            player_id=player_id,
            season=season,
            season_type_all_star=SEASON_TYPE,
        ).get_data_frames()[0]
        frame["SEASON_STR"] = season
        season_frames.append(frame)
        time.sleep(RATE_LIMIT_SLEEP_SECONDS)

    combined = pd.concat(season_frames, ignore_index=True)
    combined["GAME_DATE"] = pd.to_datetime(combined["GAME_DATE"])
    combined = combined.sort_values("GAME_DATE").reset_index(drop=True)
    return combined

In [4]:
player_id = get_player_id_exact(PLAYER_NAME)

player_info_df = commonplayerinfo.CommonPlayerInfo(player_id=player_id).get_data_frames()[0]
from_year = int(player_info_df["FROM_YEAR"].iloc[0])
to_year = int(player_info_df["TO_YEAR"].iloc[0])

season_strings = build_season_strings(from_year, to_year)
curry_games_df = fetch_career_regular_season_logs(player_id=player_id, season_strings=season_strings)

print(f"Player ID: {player_id}")
print(f"Season span: {season_strings[0]} -> {season_strings[-1]} ({len(season_strings)} seasons)")
print(f"Regular-season games fetched: {len(curry_games_df):,}")
print(f"Date range: {curry_games_df['GAME_DATE'].min().date()} -> {curry_games_df['GAME_DATE'].max().date()}")

[1/17] fetching 2009-10
[2/17] fetching 2010-11
[3/17] fetching 2011-12
[4/17] fetching 2012-13
[5/17] fetching 2013-14
[6/17] fetching 2014-15
[7/17] fetching 2015-16
[8/17] fetching 2016-17
[9/17] fetching 2017-18
[10/17] fetching 2018-19
[11/17] fetching 2019-20
[12/17] fetching 2020-21
[13/17] fetching 2021-22
[14/17] fetching 2022-23
[15/17] fetching 2023-24
[16/17] fetching 2024-25
[17/17] fetching 2025-26
Player ID: 201939
Season span: 2009-10 -> 2025-26 (17 seasons)
Regular-season games fetched: 1,065
Date range: 2009-10-28 -> 2026-01-30


In [5]:
career_summary = pd.Series(
    {
        "games": int(len(curry_games_df)),
        "avg_pts": float(curry_games_df["PTS"].mean()),
        "avg_ast": float(curry_games_df["AST"].mean()),
        "avg_reb": float(curry_games_df["REB"].mean()),
        "avg_fg3m": float(curry_games_df["FG3M"].mean()),
    }
).round(2)

career_summary

games       1065.00
avg_pts       24.83
avg_ast        6.32
avg_reb        4.65
avg_fg3m       3.97
dtype: float64

In [6]:
output_path = Path("/Users/thomasmyles/dev/betting/tmp/stephen_curry_career_regular_season_box_scores.csv")
curry_games_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

Saved: /Users/thomasmyles/dev/betting/tmp/stephen_curry_career_regular_season_box_scores.csv


In [7]:
display_columns = [
    "GAME_DATE",
    "MATCHUP",
    "WL",
    "MIN",
    "PTS",
    "REB",
    "AST",
    "FGM",
    "FGA",
    "FG3M",
    "FG3A",
    "FTM",
    "FTA",
    "PLUS_MINUS",
    "SEASON_ID",
    "SEASON_STR",
]

curry_games_df[display_columns].tail(20)

,GAME_DATE,MATCHUP,WL,MIN,PTS,REB,AST,FGM,FGA,FG3M,FG3A,FTM,FTA,PLUS_MINUS,SEASON_ID,SEASON_STR
1045,2025-12-20,GSW vs. PHX,W,35,28,10,6,9,19,4,11,6,7,13,22025,2025-26
1046,2025-12-22,GSW vs. ORL,W,31,26,3,6,10,23,4,13,2,2,17,22025,2025-26
1047,2025-12-25,GSW vs. DAL,W,33,23,3,4,6,18,2,10,9,9,-3,22025,2025-26
1048,2025-12-28,GSW @ TOR,L,41,39,3,4,12,30,4,11,11,11,0,22025,2025-26
1049,2025-12-29,GSW @ BKN,W,29,27,2,5,8,15,5,12,6,7,-6,22025,2025-26
1050,2025-12-31,GSW @ CHA,W,33,26,2,4,9,16,5,10,3,3,11,22025,2025-26
1051,2026-01-03,GSW vs. UTA,W,34,31,2,5,8,18,6,12,9,9,5,22025,2025-26
1052,2026-01-05,GSW @ LAC,L,34,27,4,6,9,23,4,15,5,5,5,22025,2025-26
1053,2026-01-07,GSW vs. MIL,W,34,31,6,7,12,21,3,9,4,5,16,22025,2025-26
1054,2026-01-09,GSW vs. SAC,W,32,27,0,10,10,21,6,12,1,1,14,22025,2025-26


In [9]:
curry_games_df.columns

Index(['SEASON_ID', 'Player_ID', 'Game_ID', 'GAME_DATE', 'MATCHUP', 'WL',
       'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA',
       'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF',
       'PTS', 'PLUS_MINUS', 'VIDEO_AVAILABLE', 'SEASON_STR'],
      dtype='object')

In [ ]:
# running this:

# python "tmp/fetch_steph_curry_career_box_scores.py" --player "Stephen Curry" --include-player-info

In [13]:
# Compare columns available in tmp export vs columns currently in your notebook DataFrame.
# Assumes your notebook DataFrame is named `curry_games_df`.
# If your df has a different name, replace it below.

import pandas as pd

columns_inventory_path = "/Users/thomasmyles/dev/betting/tmp/stephen_curry_career_box_scores_columns.csv"

inventory_df = pd.read_csv(columns_inventory_path)
available_columns = set(inventory_df["column_name"].tolist())
current_columns = set(curry_games_df.columns.tolist())

missing_in_notebook = sorted(available_columns - current_columns)
already_in_notebook = sorted(available_columns & current_columns)

print(f"Available in tmp export: {len(available_columns)}")
print(f"Currently in notebook df: {len(current_columns)}")
print(f"Missing in notebook: {len(missing_in_notebook)}")

print("\n--- Missing columns (copy from here) ---")
for col in missing_in_notebook:
    print(col)

# Optional quick view
pd.DataFrame({"missing_column": missing_in_notebook}).head(50)

Available in tmp export: 61
Currently in notebook df: 28
Missing in notebook: 33

--- Missing columns (copy from here) ---
PLAYER_INFO_BIRTHDATE
PLAYER_INFO_COUNTRY
PLAYER_INFO_DISPLAY_FIRST_LAST
PLAYER_INFO_DISPLAY_FI_LAST
PLAYER_INFO_DISPLAY_LAST_COMMA_FIRST
PLAYER_INFO_DLEAGUE_FLAG
PLAYER_INFO_DRAFT_NUMBER
PLAYER_INFO_DRAFT_ROUND
PLAYER_INFO_DRAFT_YEAR
PLAYER_INFO_FIRST_NAME
PLAYER_INFO_FROM_YEAR
PLAYER_INFO_GAMES_PLAYED_CURRENT_SEASON_FLAG
PLAYER_INFO_GAMES_PLAYED_FLAG
PLAYER_INFO_GREATEST_75_FLAG
PLAYER_INFO_HEIGHT
PLAYER_INFO_JERSEY
PLAYER_INFO_LAST_AFFILIATION
PLAYER_INFO_LAST_NAME
PLAYER_INFO_NBA_FLAG
PLAYER_INFO_PERSON_ID
PLAYER_INFO_PLAYERCODE
PLAYER_INFO_PLAYER_SLUG
PLAYER_INFO_POSITION
PLAYER_INFO_ROSTERSTATUS
PLAYER_INFO_SCHOOL
PLAYER_INFO_SEASON_EXP
PLAYER_INFO_TEAM_ABBREVIATION
PLAYER_INFO_TEAM_CITY
PLAYER_INFO_TEAM_CODE
PLAYER_INFO_TEAM_ID
PLAYER_INFO_TEAM_NAME
PLAYER_INFO_TO_YEAR
PLAYER_INFO_WEIGHT


,missing_column
0,PLAYER_INFO_BIRTHDATE
1,PLAYER_INFO_COUNTRY
2,PLAYER_INFO_DISPLAY_FIRST_LAST
3,PLAYER_INFO_DISPLAY_FI_LAST
4,PLAYER_INFO_DISPLAY_LAST_COMMA_FIRST
5,PLAYER_INFO_DLEAGUE_FLAG
6,PLAYER_INFO_DRAFT_NUMBER
7,PLAYER_INFO_DRAFT_ROUND
8,PLAYER_INFO_DRAFT_YEAR
9,PLAYER_INFO_FIRST_NAME


In [14]:
inventory_df

,column_name,dtype,non_null_count
0,SEASON_ID,object,1065
1,Player_ID,int64,1065
2,Game_ID,object,1065
3,GAME_DATE,datetime64[ns],1065
4,MATCHUP,object,1065
...,...,...,...
56,PLAYER_INFO_GAMES_PLAYED_FLAG,object,1065
57,PLAYER_INFO_DRAFT_YEAR,object,1065
58,PLAYER_INFO_DRAFT_ROUND,object,1065
59,PLAYER_INFO_DRAFT_NUMBER,object,1065


In [15]:
# doing the above in here:

import subprocess
import pandas as pd

subprocess.run(
    [
        "python",
        "/Users/thomasmyles/dev/betting/tmp/fetch_steph_curry_career_box_scores.py",
        "--player",
        "Stephen Curry",
        "--include-player-info",
    ],
    check=True,
)

curry_games_df = pd.read_csv(
    "/Users/thomasmyles/dev/betting/tmp/stephen_curry_career_box_scores.csv"
)
curry_games_df["GAME_DATE"] = pd.to_datetime(curry_games_df["GAME_DATE"])

print("shape:", curry_games_df.shape)
print("column_count:", len(curry_games_df.columns))
print("player_info_cols:", len([c for c in curry_games_df.columns if c.startswith("PLAYER_INFO_")]))
curry_games_df.head(3)

[1/17] fetching 2009-10
[2/17] fetching 2010-11
[3/17] fetching 2011-12
[4/17] fetching 2012-13
[5/17] fetching 2013-14
[6/17] fetching 2014-15
[7/17] fetching 2015-16
[8/17] fetching 2016-17
[9/17] fetching 2017-18
[10/17] fetching 2018-19
[11/17] fetching 2019-20
[12/17] fetching 2020-21
[13/17] fetching 2021-22
[14/17] fetching 2022-23
[15/17] fetching 2023-24
[16/17] fetching 2024-25
[17/17] fetching 2025-26

player_id: 201939
season_span: 2009-10 -> 2025-26 (17 seasons, Regular Season)
games: 1065
avg_pts/ast/reb: 24.83/6.32/4.65
column_count: 61
saved_csv: /Users/thomasmyles/dev/betting/tmp/stephen_curry_career_box_scores.csv
saved_columns_inventory: /Users/thomasmyles/dev/betting/tmp/stephen_curry_career_box_scores_columns.csv
shape: (1065, 61)
column_count: 61
player_info_cols: 33


,SEASON_ID,Player_ID,Game_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,...,PLAYER_INFO_PLAYERCODE,PLAYER_INFO_FROM_YEAR,PLAYER_INFO_TO_YEAR,PLAYER_INFO_DLEAGUE_FLAG,PLAYER_INFO_NBA_FLAG,PLAYER_INFO_GAMES_PLAYED_FLAG,PLAYER_INFO_DRAFT_YEAR,PLAYER_INFO_DRAFT_ROUND,PLAYER_INFO_DRAFT_NUMBER,PLAYER_INFO_GREATEST_75_FLAG
0,22009,201939,20900015,2009-10-28,GSW vs. HOU,L,36,7,12,0.583,...,stephen_curry,2009,2025,N,Y,Y,2009,1,7,Y
1,22009,201939,20900030,2009-10-30,GSW @ PHX,L,39,5,9,0.556,...,stephen_curry,2009,2025,N,Y,Y,2009,1,7,Y
2,22009,201939,20900069,2009-11-04,GSW vs. MEM,W,28,3,6,0.500,...,stephen_curry,2009,2025,N,Y,Y,2009,1,7,Y


In [16]:
curry_games_df

,SEASON_ID,Player_ID,Game_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,...,PLAYER_INFO_PLAYERCODE,PLAYER_INFO_FROM_YEAR,PLAYER_INFO_TO_YEAR,PLAYER_INFO_DLEAGUE_FLAG,PLAYER_INFO_NBA_FLAG,PLAYER_INFO_GAMES_PLAYED_FLAG,PLAYER_INFO_DRAFT_YEAR,PLAYER_INFO_DRAFT_ROUND,PLAYER_INFO_DRAFT_NUMBER,PLAYER_INFO_GREATEST_75_FLAG
0,22009,201939,20900015,2009-10-28,GSW vs. HOU,L,36,7,12,0.583,...,stephen_curry,2009,2025,N,Y,Y,2009,1,7,Y
1,22009,201939,20900030,2009-10-30,GSW @ PHX,L,39,5,9,0.556,...,stephen_curry,2009,2025,N,Y,Y,2009,1,7,Y
2,22009,201939,20900069,2009-11-04,GSW vs. MEM,W,28,3,6,0.500,...,stephen_curry,2009,2025,N,Y,Y,2009,1,7,Y
3,22009,201939,20900082,2009-11-06,GSW vs. LAC,L,22,1,5,0.200,...,stephen_curry,2009,2025,N,Y,Y,2009,1,7,Y
4,22009,201939,20900096,2009-11-08,GSW @ SAC,L,31,4,8,0.500,...,stephen_curry,2009,2025,N,Y,Y,2009,1,7,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1060,22025,201939,22500618,2026-01-20,GSW vs. TOR,L,25,6,16,0.375,...,stephen_curry,2009,2025,N,Y,Y,2009,1,7,Y
1061,22025,201939,22500630,2026-01-22,GSW @ DAL,L,34,14,27,0.519,...,stephen_curry,2009,2025,N,Y,Y,2009,1,7,Y
1062,22025,201939,22500644,2026-01-25,GSW @ MIN,W,28,7,18,0.389,...,stephen_curry,2009,2025,N,Y,Y,2009,1,7,Y
1063,22025,201939,22500678,2026-01-28,GSW @ UTA,W,28,7,14,0.500,...,stephen_curry,2009,2025,N,Y,Y,2009,1,7,Y


In [17]:
curry_games_df.to_dict('records')[0]

{'SEASON_ID': 22009,
 'Player_ID': 201939,
 'Game_ID': 20900015,
 'GAME_DATE': Timestamp('2009-10-28 00:00:00'),
 'MATCHUP': 'GSW vs. HOU',
 'WL': 'L',
 'MIN': 36,
 'FGM': 7,
 'FGA': 12,
 'FG_PCT': 0.583,
 'FG3M': 0,
 'FG3A': 1,
 'FG3_PCT': 0.0,
 'FTM': 0,
 'FTA': 0,
 'FT_PCT': 0.0,
 'OREB': 1,
 'DREB': 1,
 'REB': 2,
 'AST': 7,
 'STL': 4,
 'BLK': 0,
 'TOV': 2,
 'PF': 2,
 'PTS': 14,
 'PLUS_MINUS': 7,
 'VIDEO_AVAILABLE': 0,
 'SEASON_STR': '2009-10',
 'PLAYER_INFO_PERSON_ID': 201939,
 'PLAYER_INFO_FIRST_NAME': 'Stephen',
 'PLAYER_INFO_LAST_NAME': 'Curry',
 'PLAYER_INFO_DISPLAY_FIRST_LAST': 'Stephen Curry',
 'PLAYER_INFO_DISPLAY_LAST_COMMA_FIRST': 'Curry, Stephen',
 'PLAYER_INFO_DISPLAY_FI_LAST': 'S. Curry',
 'PLAYER_INFO_PLAYER_SLUG': 'stephen-curry',
 'PLAYER_INFO_BIRTHDATE': '1988-03-14T00:00:00',
 'PLAYER_INFO_SCHOOL': 'Davidson',
 'PLAYER_INFO_COUNTRY': 'USA',
 'PLAYER_INFO_LAST_AFFILIATION': 'Davidson/USA',
 'PLAYER_INFO_HEIGHT': '6-2',
 'PLAYER_INFO_WEIGHT': 185,
 'PLAYER_INFO_SEASO

In [18]:
curry_games_df.to_dict('records')[-1]

{'SEASON_ID': 22025,
 'Player_ID': 201939,
 'Game_ID': 22500696,
 'GAME_DATE': Timestamp('2026-01-30 00:00:00'),
 'MATCHUP': 'GSW vs. DET',
 'WL': 'L',
 'MIN': 25,
 'FGM': 7,
 'FGA': 16,
 'FG_PCT': 0.438,
 'FG3M': 4,
 'FG3A': 10,
 'FG3_PCT': 0.4,
 'FTM': 5,
 'FTA': 5,
 'FT_PCT': 1.0,
 'OREB': 0,
 'DREB': 1,
 'REB': 1,
 'AST': 2,
 'STL': 0,
 'BLK': 1,
 'TOV': 4,
 'PF': 0,
 'PTS': 23,
 'PLUS_MINUS': -5,
 'VIDEO_AVAILABLE': 1,
 'SEASON_STR': '2025-26',
 'PLAYER_INFO_PERSON_ID': 201939,
 'PLAYER_INFO_FIRST_NAME': 'Stephen',
 'PLAYER_INFO_LAST_NAME': 'Curry',
 'PLAYER_INFO_DISPLAY_FIRST_LAST': 'Stephen Curry',
 'PLAYER_INFO_DISPLAY_LAST_COMMA_FIRST': 'Curry, Stephen',
 'PLAYER_INFO_DISPLAY_FI_LAST': 'S. Curry',
 'PLAYER_INFO_PLAYER_SLUG': 'stephen-curry',
 'PLAYER_INFO_BIRTHDATE': '1988-03-14T00:00:00',
 'PLAYER_INFO_SCHOOL': 'Davidson',
 'PLAYER_INFO_COUNTRY': 'USA',
 'PLAYER_INFO_LAST_AFFILIATION': 'Davidson/USA',
 'PLAYER_INFO_HEIGHT': '6-2',
 'PLAYER_INFO_WEIGHT': 185,
 'PLAYER_INFO_SEA